# 不同 State：在节点内调用子图并转换 state

**场景：检索管线复用**。检索团队把"搜索 → 重排"封装成独立子图，它有自己的 schema
（`query`/`hits`）；现在要嵌进"周报生成"父图，父图 schema 是 `topic`/`report`——零共享 key。

这种**组件边界清晰**的子图正是节点内调用的主场：把子图当普通函数在节点里 `invoke`，
手动完成两处映射（父图字段 → 子图入参；子图结果 → 父图字段）。附带收益：
检索子图可以单独测试、被多个父图复用，映射处还能加日志/校验等附加逻辑。

In [3]:
from langgraph.graph import END, START, StateGraph
from rich import print
from typing_extensions import TypedDict


# ---- 检索子图：检索团队维护，自有 schema（query/hits）----
class RetrievedState(TypedDict):
    query: str
    hits: list[str]


DOCS = [
    "LangGraph 子图把图作为节点复用",
    "LangGraph 中断实现人工审批",
    "LangGraph 流式输出逐节点推送",
    "Python 装饰器原理与实践",
    "SQL 索引优化指南",
]


def search(state: RetrievedState):
    words = state["query"].split()
    hits = [d for d in DOCS if any(w in d for w in words)]
    return {"hits": hits}


def rerank(state: RetrievedState):
    # 重排：短文档排前面（模拟相关性打分）
    return {"hits": sorted(state["hits"], key=len)}


retriever = (StateGraph(RetrievedState)
             .add_node("search", search)
             .add_node("rerank", rerank)
             .add_edge(START, "search")
             .add_edge("search", "rerank")
             .add_edge("rerank", END)
             .compile())   # 可独立测试：retriever.invoke({"query": "LangGraph 中断"})


# ---- 周报父图：自有 schema（topic/report），与子图零重叠 ----
class ReportState(TypedDict):
    topic: str
    report: str


def retrieve(state: ReportState):
    # 父 -> 子：topic 翻译成检索词
    resp = retriever.invoke({"query": state["topic"]})
    # 子 -> 父：hits 拼成周报要点
    return {"report": "本周要点：\n- " + "\n- ".join(resp["hits"])}


builder = StateGraph(ReportState)
builder.add_node("retrieve", retrieve)
builder.add_edge(START, "retrieve")
graph = builder.compile()

print(graph.invoke({"topic": "LangGraph 中断"}))

# 检索子图可单独测试（组件化的意义）
print(retriever.invoke({"query": "LangGraph 子图"}))

{
    'topic': 'LangGraph 中断',
    'report': '本周要点：\n- LangGraph 中断实现人工审批\n- LangGraph 流式输出逐节点推送\n- LangGraph 
子图把图作为节点复用'
}

{
    'query': 'LangGraph 子图',
    'hits': ['LangGraph 中断实现人工审批', 'LangGraph 流式输出逐节点推送', 'LangGraph 子图把图作为节点复用']
}

## 踩坑：不同 schema 的子图直接作节点——数据悄悄断了线

把检索子图直接传给 `add_node` 会怎样？LangGraph 按子图 schema 过滤父图 state：
`topic` 不在子图 schema 里 → 子图收到**空输入**（检索词丢失）。
本例 `search` 节点读 `state["query"]`，直接在子图内部 `KeyError`——报错位置在子图深处，
极易误判成检索子图自身的 bug；而如果子图节点不读缺失 key，错误更隐蔽：
子图正常跑完，输出 `hits` 不在父图 schema 被**静默丢弃**，`report` 永远是空的，全程无报错。

In [4]:
parent = (StateGraph(ReportState)
          .add_node("retrieve", retriever)   # 零共享 key 的子图直接作节点
          .add_edge(START, "retrieve")
          .compile())

# 子图收到空输入 {}：search 读不到 query，在子图内部 KeyError
try:
    parent.invoke({"topic": "LangGraph 中断"})
except KeyError as e:
    print(f"子图内部 KeyError: {e}")

子图内部 KeyError: 'query'

## 实测结论

1. **零共享 key 时只能节点内调用**：进出的两处映射显式可见，数据流一目了然。
2. **组件化收益**：子图 schema 自洽（`query`/`hits`），可独立测试、多父图复用。
3. **不同 schema 直接作节点是坑**（实测 1.2.11）：子图收到空输入，读缺失 key 则子图内部 `KeyError`（易误判），不读则输出被静默丢弃、父图字段不更新。
4. 两种模式的选择：有共享 key 且无需额外逻辑 → 直接作节点（[01](01_共享State：子图直接作节点.ipynb)）；
   无共享 key、需要转换/附加逻辑 → 节点内调用。